In [10]:
import numpy as np
from scipy import signal

In [11]:
# we shall use plotly for plotting functions
# we use plotly.express instead of plotly.graph_objects for better performance and less boilerplate
import plotly.express as px

## introduction to fourier series

> Fourier showed that any function, $\ell(t)$, defined in the interval
> $t \in (0,\pi)$, could be expressed as an infinite linear combination of
> harmonically related sinusoids,
>
> $$\ell(t) = a_1 \sin(t) + a_2 \sin(2t) + a_3 \sin(3t) + \dots$$
>
> and that the value of the coefficients $a_n$ could be computed as the area
> of the curve $\ell(t)\sin(nt)$. Precisely,
>
> $$a_n = \frac{2}{\pi} \int_0^\pi \ell(t) \sin(nt)\, dt$$
>
> However, the sum is only guaranteed to converge to the function $\ell(t)$
> within the interval $t \in (0,\pi)$. In fact, the resulting sum, for any
> values $a_n$, is a **periodic function** with period $2\pi$ and is
> anti-symmetric with respect to the origin, $t = 0$.

> One of Fourier's original examples of **sine series** is the expansion
> of the ramp signal $\ell(t) = t/2$. This series was first introduced by
> Euler. Fourier showed that his theory explained why a ramp could be
> written as the following infinite sum:
>
> $$\frac{1}{2} t = \sin(t) - \frac{1}{2} \sin(2t) + \frac{1}{3} \sin(3t) - \frac{1}{4} \sin(4t) + \dots$$
>
> The result of this series approximates the ramp with increasing accuracy
> as we add more terms.

lets try to recreate Fourier's ramp function approximation implementation in code

In [12]:
# lets just use 0 to pi to simplify integration, otherwise we would need to cut or extend the
# list mid-way through calculation
x = np.linspace(0, np.pi, 128)

# lets create a ramp signal here, and lets use a lambda expression to make things easier to read
ramp_signal = lambda x: 0.5 * x

# til: you dont need to use [ramp_signal(_x) for _x in x] to return a list of y values,
# just do ramp_signal(x) since 0.5 * x already works for lists (nice)
graph = px.line(x = x, y = ramp_signal(x))
graph.show()

In [13]:
# lets use scipy.integrate.simpson instead of manually doing integration on samples
import scipy.integrate as integrate

# now lets create the approximation function
def fourier_approx(x: list, y: list, accuracy: int) -> list:
  res = np.zeros(len(x))

  # a bit of a syntax tounge twister here in the for loop, 
  # 0->4 is range(accuracy), but 1->5 is range(1, accuracy + 1)
  for i in range(1, accuracy + 1):
    # print(f"{i=}")

    a = (2 / np.pi) * integrate.simpson(x = x, y = y * np.sin(i * x))
    # print(f"{a=}")

    res = res + (a * np.sin(i * x))
    # print(f"{res=}")
  return res

graph = px.line(x = x, y = fourier_approx(x = x, y = ramp_signal(x), accuracy = 16))
graph.add_traces(px.line(x = x, y = ramp_signal(x)).data)

graph.show()

as we can see, this creates a very neat way to approximate a function using sine waves, granted we only look at the interval $t \in (0,\pi)$, but this fundamental idea is great! even seemingly random function can be approximated as a collection of sine waves

In [14]:
rng = np.random.default_rng(67)

random_signal = lambda x: rng.normal(0, 1)
random_y = [random_signal(_x) for _x in x]
# we expand to list here ([random_signal(_x) for _x in x]) since random.randint() cant take a list
graph = px.line(
  x = x, 
  y = fourier_approx(x = x, y = random_y, accuracy = 32))
graph.update_traces(line_color="red")

graph.add_traces(px.line(x = x, y = random_y).data)
graph.show()